In [ ]:
from data_processing import *
from kernels import *
from methods import *
from utils import *
from features import *
from data_augmentation import *

In [ ]:
# Load data
Xtr, Xte, Ytr = load_data(data_dir="../")
# Preprocess data
std_Xtr, Xtr, Xte = normalize_data(Xtr, Xte)
X_train, Y_train = data_transformations(Xtr, Ytr, only_flip=True)
# Hog features
X_hog_train = extract_hog_features(X_train, cell_size=8, bins=9, block_size=2)
X_hog_test = extract_hog_features(Xte, cell_size=8, bins=9, block_size=2)
# Y labels
Y_train_labels = 2*one_hot_encode(Y_train, 10)-1

In [ ]:
# Kernels 
RBF = RBFKernel(gamma=1)
Chi2 = ChiSquareKernel(gamma=0.2) 
# Train kernels 
K_train_rbf_hog = RBF(X_hog_train, X_hog_train)
K_train_chi_hog = Chi2(X_hog_train, X_hog_train, block_size=128)
# Test kernels
K_test_rbf_hog = RBF(X_hog_test, X_hog_train)
K_test_chi_hog = Chi2(X_hog_test, X_hog_train, block_size=128)
# For normalization 
K_test_rbf_hog_same = RBF(X_hog_test, X_hog_test)
K_test_chi_hog_same = Chi2(X_hog_test, X_hog_test, block_size=128)

In [ ]:
def normalize_kernel(K):
    d = np.sqrt(np.diag(K))
    return K / (d[:,None] * d[None,:] + 1e-12)

def normalize_kernel_val(K, K_val, K_train):
    d_val = np.sqrt(np.diag(K_val))
    d_train = np.sqrt(np.diag(K_train))
    return K / (d_val[:,None] * d_train[None,:] + 1e-12)

In [ ]:
# Normalized kernels
K_train_rbf_hog = normalize_kernel(K_train_rbf_hog)
K_train_chi_hog = normalize_kernel(K_train_chi_hog)
K_test_rbf_hog = normalize_kernel_val(K_test_rbf_hog, K_test_rbf_hog_same, K_train_rbf_hog)
K_test_chi_hog = normalize_kernel_val(K_test_chi_hog, K_test_chi_hog_same, K_train_chi_hog)
# Final kernels
K_train = 0.25*K_train_rbf_hog + 0.75*K_train_chi_hog
K_test = 0.25*K_test_rbf_hog + 0.75*K_test_chi_hog

In [ ]:
model = SVM(K_train)
alphas, b = model.fit(Y_train_labels)
Y_pred_val = model.predict(K_test, Y_train_labels, alphas, b)
submission_df = format_submission(Y_pred_val)